<!-- codex-architecture-notes -->
## Architectural Notes

**Purpose:** Runs the reusable feature-engineering job against the selected modeling population and inspects GPA repair outputs.

**Inputs / Data Sources:**
- `SELECTED_MODEL_POPULATION_PATH` from `src.paths` (written by `01_select_model_population`)

**Outputs / Side Effects:**
- `FEATURE_ENGINEERED_PRIMARY_PATH` from `src.paths` (sole owner; saves `df_primary`)

**Logic Flow:**
1. Load the selected modeling population (data-root guard first).
2. Run `run_feature_engineering_job`.
3. Inspect primary, audit, and excluded rows.
4. Save `df_primary` for the split stage.

**Maintainability Notes:** Import constants from `src.paths`; diploma columns pass through the job untouched (verified — no generic column-wide transform).

In [1]:
import pandas as pd 

In [2]:
pd.set_option('display.max_columns',100)

In [3]:
import sys
from pathlib import Path

bootstrap_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src').is_dir()
)
if str(bootstrap_root) not in sys.path:
    sys.path.insert(0, str(bootstrap_root))

from src.paths import (
    FEATURE_ENGINEERED_PRIMARY_PATH,
    SELECTED_MODEL_POPULATION_PATH,
    assert_data_root,
)
from src.io_utils import save_parquet
from src.feature_engineering import run_feature_engineering_job, assert_no_leakage_columns

In [4]:
assert_data_root(SELECTED_MODEL_POPULATION_PATH)
df = pd.read_parquet(SELECTED_MODEL_POPULATION_PATH)

In [5]:
df.shape


(784245, 28)

In [6]:
df.head()

,student_course_id,student_id,course_id,part_id,degree_id,faculty_id,grade_id,final_mark,course_credits,attempt_number,student_status_id,prev_gpa_points,gpa_points,start_agpa_points,start_total_in_courses,start_total_in_credits,semester_reg_credits,semester_reg_courses,semester_pass_credits,total_pass_credits,total_fail_credits,reg_total_semesters,start_level_name_pl,start_part_id,requirement_type_id,degree_requirement_credits_count,diploma_gpa,diploma_type_id
0,939272.111,10000.111,1016.111,20152,3.111,5.111,677.111,82,2.0,1,155348.111,2.41,2.36,2.41,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,2,6,74.71,15
1,953865.111,10000.111,1017.111,20153,3.111,5.111,680.111,67,2.0,1,155349.111,2.36,2.25,2.39,12.0,35.0,9.0,3.0,9.0,35.0,0.0,3.0,Second Year,20151,2,6,74.71,15
2,1076397.111,10000.111,1019.111,20171,3.111,5.111,981.111,66,2.0,1,240678.111,2.17,1.76,2.28,27.0,80.0,17.0,6.0,17.0,80.0,0.0,6.0,Fourth Year,20151,2,6,74.71,15
3,925010.111,10000.111,431.111,20152,3.111,5.111,677.111,83,3.0,1,155348.111,2.41,2.36,2.41,6.0,17.0,18.0,6.0,18.0,17.0,0.0,2.0,Second Year,20151,3,65,74.71,15
4,901340.111,10000.111,432.111,20151,3.111,5.111,681.111,61,4.0,1,155347.111,<NA>,2.41,0.0,0.0,0.0,17.0,6.0,17.0,0.0,0.0,1.0,First Year,20151,3,65,74.71,15


In [7]:
df.columns

Index(['student_course_id', 'student_id', 'course_id', 'part_id', 'degree_id',
       'faculty_id', 'grade_id', 'final_mark', 'course_credits',
       'attempt_number', 'student_status_id', 'prev_gpa_points', 'gpa_points',
       'start_agpa_points', 'start_total_in_courses', 'start_total_in_credits',
       'semester_reg_credits', 'semester_reg_courses', 'semester_pass_credits',
       'total_pass_credits', 'total_fail_credits', 'reg_total_semesters',
       'start_level_name_pl', 'start_part_id', 'requirement_type_id',
       'degree_requirement_credits_count', 'diploma_gpa', 'diploma_type_id'],
      dtype='str')

In [8]:
result = run_feature_engineering_job(
    df,
    structural_zero_as_nan=True
)

Original df rows: 784245
Suffix consistency check: no conflicts detected.
No semester-level conflicts detected before aggregation.
Timeline diagnostics:


,value
student_degree_timeline_count,16935.0
part_sort_key_min,20051.0
part_sort_key_max,20253.0
first_semester_concept_mismatch_count,1150.0


Semester feature merge check:


,row_count
_merge,
both,784245
left_only,0
right_only,0


Last valid GPA non-null ratio in semester frame: 0.8934182254225836
Last valid GPA non-null ratio after merge in df_model_audit: 0.8728178056602209
Previous-GPA chain diagnostics:


...

,conflict_count
prev_gpa_points_clean,0
prev_gpa_fill_source,0
last_valid_gpa_before_current_semester,0
is_interruption_semester,0
prev_semester_was_interruption,0
prior_interruption_count,0
consecutive_interruption_count,0


Semester stability check: all columns stable within semester groups.
Last valid GPA non-null ratio in df_primary: 0.8677652320208986
Row counts:


,row_count
original_df,784245
df_model_audit,784245
df_primary,749523
df_excluded_over_policy,34722


Final added or repaired columns:


,column
0,university_id
1,is_high_credit_course
2,over_policy_semester_credits
3,over_policy_semester_courses
4,exclude_over_policy_semester
5,is_extreme_fail_history
6,total_fail_credits_capped
7,is_interruption_semester
8,prev_semester_was_interruption
9,prior_interruption_count


In [9]:
df_model_audit = result["df_model_audit"]
df_primary = result["df_primary"]
df_excluded_over_policy = result["df_excluded_over_policy"]
diagnostics = result["diagnostics"]

In [10]:
diagnostics["row_counts"]

{'original_df': 784245,
 'df_model_audit': 784245,
 'df_primary': 749523,
 'df_excluded_over_policy': 34722}

In [11]:
diagnostics["prev_gpa_fill_source_counts"]

,row_count
prev_gpa_fill_source,
raw_prev_gpa,654742
zero_fallback,95816
last_valid_gpa_before_current_semester,32899
start_agpa_points,788


In [12]:
diagnostics["timeline_diagnostics"]

{'student_degree_timeline_count': 16935,
 'part_sort_key_min': 20051.0,
 'part_sort_key_max': 20253.0,
 'first_semester_concept_mismatch_count': 1150}

In [13]:
diagnostics["start_level_ord_counts"]

start_level_ord
1    191965
2    174156
3    148578
4    123350
5    119066
6     27130
Name: count, dtype: int64

In [14]:
diagnostics["suspicious_zero_fallback_count"]

414

In [15]:
diagnostics["advanced_standing_zero_fallback_count"]

2404

In [16]:
diagnostics["semester_stability_conflicts"]

{'prev_gpa_points_clean': 0,
 'prev_gpa_fill_source': 0,
 'last_valid_gpa_before_current_semester': 0,
 'is_interruption_semester': 0,
 'prev_semester_was_interruption': 0,
 'prior_interruption_count': 0,
 'consecutive_interruption_count': 0}

In [17]:
diagnostics["advanced_standing_zero_fallback_rows"].head(20)

,university_id,student_id,degree_id,part_id,course_id,prev_gpa_points,last_valid_gpa_before_current_semester,start_agpa_points,prev_gpa_points_clean,prev_gpa_fill_source,is_first_active_semester,is_first_row_in_timeline,no_previous_progress,start_level_ord
1023,111,10018.111,2.111,20151,657.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
1030,111,10018.111,2.111,20151,665.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
1035,111,10018.111,2.111,20151,672.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
1099,111,10018.111,2.111,20151,955.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
1100,111,10018.111,2.111,20151,962.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
3422,111,10050.111,1.111,20151,1038.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
3423,111,10050.111,1.111,20151,296.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
3430,111,10050.111,1.111,20151,298.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
3431,111,10050.111,1.111,20151,299.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2
3439,111,10050.111,1.111,20151,302.111,<NA>,<NA>,0.0,0.0,zero_fallback,1,1,1,2


In [18]:
len(diagnostics)

20

In [19]:
primary = result["df_primary"]


In [20]:
len(primary)

749523

In [21]:
primary.columns

Index(['student_course_id', 'student_id', 'course_id', 'part_id', 'degree_id',
       'faculty_id', 'grade_id', 'final_mark', 'course_credits',
       'attempt_number', 'student_status_id', 'prev_gpa_points', 'gpa_points',
       'start_agpa_points', 'start_total_in_courses', 'start_total_in_credits',
       'semester_reg_credits', 'semester_reg_courses', 'semester_pass_credits',
       'total_pass_credits', 'total_fail_credits', 'reg_total_semesters',
       'start_level_name_pl', 'start_part_id', 'requirement_type_id',
       'degree_requirement_credits_count', 'diploma_gpa', 'diploma_type_id',
       'university_id', 'is_high_credit_course',
       'over_policy_semester_credits', 'over_policy_semester_courses',
       'exclude_over_policy_semester', 'is_extreme_fail_history',
       'total_fail_credits_capped', 'is_interruption_semester',
       'prev_semester_was_interruption', 'prior_interruption_count',
       'consecutive_interruption_count', 'no_previous_progress',
       'is_f

In [22]:
primary['last_valid_gpa_before_current_semester']

0         2.41
1         2.36
2         2.17
3         2.41
4         <NA>
          ... 
784240    <NA>
784241    0.86
784242    1.75
784243    2.58
784244    <NA>
Name: last_valid_gpa_before_current_semester, Length: 749523, dtype: Float64

In [23]:
history_cols = [
    "last_valid_gpa_before_current_semester",
    "gpa_trend_delta",
    "gpa_trend_missing",
    "is_interruption_semester",
    "prev_semester_was_interruption",
    "prior_interruption_count",
    "consecutive_interruption_count",
]

for c in history_cols:
    print("\n", c)
    print(primary[c].describe())
    print(primary[c].value_counts(dropna=False).head(10))


 last_valid_gpa_before_current_semester
count    650410.0
mean     2.188716
std      0.764438
min          0.08
25%          1.71
50%          2.28
75%          2.75
max           4.0
Name: last_valid_gpa_before_current_semester, dtype: Float64
last_valid_gpa_before_current_semester
<NA>    99113
2.5     13148
2.25    13112
2.0     11716
2.75    11611
3.0      8987
1.75     8656
1.5      7663
2.38     6744
2.63     6414
Name: count, dtype: Int64

 gpa_trend_delta
count    567294.0
mean     0.009036
std      0.631759
min         -3.89
25%         -0.36
50%           0.0
75%          0.38
max           3.4
Name: gpa_trend_delta, dtype: Float64
gpa_trend_delta
 <NA>    182229
 0.0       6268
-0.25      4507
 0.25      4395
 0.04      4194
-0.04      4073
 0.02      3989
-0.08      3838
-0.06      3807
-0.02      3752
Name: count, dtype: Int64

 gpa_trend_missing
count    749523.000000
mean          0.243127
std           0.428971
min           0.000000
25%           0.000000
50%         

In [24]:
save_parquet(primary, FEATURE_ENGINEERED_PRIMARY_PATH, index=None)

WindowsPath('D:/AI/Real projects/Academic_Advisor/data/features/feature_engineered_primary.parquet')